# 04 — Role-to-Business-Line RBAC Demo

This notebook implements the **exact** algorithm behind the real `/role-check` endpoint in `app.py`
(lines ~857-908) as pure, offline Python — no FastAPI, no real Azure AD token, no network calls. It
matches the real branching logic line for line, including three genuinely surprising real quirks the
source code has that are easy to miss on a first read:

1. When `user_roles` doesn't include a mapped role at all, `sorted(business_lines)` can return an
   **empty list** — not every combination of `user_roles` produces a non-empty result.
2. **The GTRM shortcut can return `TPMB` in the result even when `FEATURE_THIRD_PARTY_BOT` is off** —
   as long as `FEATURE_GREEN_TIME` is on and the user holds all four `stitt.ingester*` roles, the GTRM
   branch returns `["IWPB", "FEMA", "TPMB", "GTRM"]` directly, without ever re-checking the TPMB feature
   flag. This is a direct reading of the real code's `if`/`elif`-free structure (two independent `if`
  blocks, not a mutually exclusive chain), not a simplification introduced by this notebook.
3. **The unconditional direct mapping (`business_lines = [...]`) doesn't check any feature flag at
   all.** It's built once, at the top, from whichever `stitt.ingester*` roles the user happens to hold
   -- so if a user is assigned `stitt.ingester.tpmb` directly (without necessarily holding the other two
   TPMB-tier roles), `TPMB` will appear in the final `sorted(business_lines)` fallback **even with
   `FEATURE_THIRD_PARTY_BOT` off**, because the flag only gates the early-return *shortcut*, not
   whether an individually-held role maps to a business line at all. In practice this likely never
   triggers, because AAD role assignment is presumably provisioned progressively (a user gets `.tpmb`
   together with the two roles beneath it) -- but it's worth being precise that the flag's real job,
   read literally, is narrower than "controls TPMB visibility."

See chapter 03 (`03-multi-department-architecture-and-business-line-routing.md`) and chapter 04
(`04-authentication-rbac-with-msal-and-oauth.md`) for the narrative explanation this notebook
demonstrates as runnable code.


## 1. The real algorithm, reproduced exactly

This is a direct line-for-line port of `app.py`'s `/role-check` handler body into a standalone,
testable function. The real handler decodes `user_roles` from an unverified JWT
(`x-ms-token-aad-id-token`, chapter 04) and reads the three feature flags from
`utils.get_feature_flags()` (an on-behalf-of call to the HSBC INM-AI Config Service) — both are passed
in here as plain arguments instead, so the *branching logic itself* can be exercised deterministically,
offline, against synthetic inputs.


In [1]:
from typing import List, Optional, Set

ROLE_TO_BUSINESS_LINE = {
    "stitt.ingester": "IWPB",
    "stitt.ingester.pilot": "FEMA",
    "stitt.ingester.tpmb": "TPMB",
    "stitt.ingester.gtrm": "GTRM",
}


def role_check(
    user_roles: List[str],
    feature_logical_seperation_value: bool,
    feature_business_line_tpmb: bool,
    feature_business_line_gtrm: bool,
) -> Optional[List[str]]:
    """
    Exact reproduction of app.py's `/role-check` handler body.

    Parameters mirror the real globals set from utils.get_feature_flags() in app.py's `/` handler:
      - feature_logical_seperation_value -> FEATURE_LOGICAL_SEPERATION
      - feature_business_line_tpmb       -> FEATURE_THIRD_PARTY_BOT
      - feature_business_line_gtrm       -> FEATURE_GREEN_TIME

    Returns None when FEATURE_LOGICAL_SEPERATION is off, exactly like the real handler -- it has no
    `return` statement at all in that branch (see FULL_ARCHITECTURE.md section 10b's flowchart: "(no
    business lines returned in this branch)"), so falling off the end of a Python function implicitly
    returns None.
    """

    if feature_logical_seperation_value:

        # Determine the business lines based on user roles
        business_lines = [
            ROLE_TO_BUSINESS_LINE[role]
            for role in user_roles
            if role in ROLE_TO_BUSINESS_LINE
        ]

        # Handle feature_business_line_tpmb logic
        if feature_business_line_tpmb:
            tpmb_roles: Set[str] = {
                "stitt.ingester",
                "stitt.ingester.pilot",
                "stitt.ingester.tpmb",
            }
            # Check if all 3 roles are present
            if tpmb_roles.issubset(user_roles):
                return ["IWPB", "FEMA", "TPMB"]

        # Handle feature_business_line_gtrm logic
        if feature_business_line_gtrm:
            gtrm_roles: Set[str] = {
                "stitt.ingester",
                "stitt.ingester.pilot",
                "stitt.ingester.tpmb",
                "stitt.ingester.gtrm",
            }
            # Check if all 4 roles are present
            if gtrm_roles.issubset(user_roles):
                return ["IWPB", "FEMA", "TPMB", "GTRM"]

        # Return the business lines based on the roles present
        return sorted(business_lines)

    # feature_logical_seperation_value is falsy -- the real handler has no return here at all.
    return None


print("role_check() defined -- exact port of app.py's /role-check handler body.")


role_check() defined -- exact port of app.py's /role-check handler body.


## 2. Exercise it against synthetic role-sets and feature-flag combinations

Each scenario below states the AAD roles a synthetic user holds and the three feature-flag values, then
prints (and asserts) the exact business-line list `/role-check` would return for that combination.


In [2]:
scenarios = [
    {
        "label": "Feature flag off entirely -- no business lines returned, regardless of roles",
        "user_roles": ["stitt.ingester", "stitt.ingester.pilot", "stitt.ingester.tpmb", "stitt.ingester.gtrm"],
        "flags": dict(feature_logical_seperation_value=False, feature_business_line_tpmb=True, feature_business_line_gtrm=True),
        "expected": None,
    },
    {
        "label": "IWPB-only role, flags on but TPMB/GTRM roles absent -- direct mapping only",
        "user_roles": ["stitt.ingester"],
        "flags": dict(feature_logical_seperation_value=True, feature_business_line_tpmb=True, feature_business_line_gtrm=True),
        "expected": ["IWPB"],
    },
    {
        "label": "IWPB + FEMA roles, TPMB role missing -- direct mapping, alphabetically sorted",
        "user_roles": ["stitt.ingester", "stitt.ingester.pilot"],
        "flags": dict(feature_logical_seperation_value=True, feature_business_line_tpmb=True, feature_business_line_gtrm=True),
        "expected": ["FEMA", "IWPB"],
    },
    {
        "label": "All 3 TPMB-tier roles present, FEATURE_THIRD_PARTY_BOT on -- TPMB shortcut fires",
        "user_roles": ["stitt.ingester", "stitt.ingester.pilot", "stitt.ingester.tpmb"],
        "flags": dict(feature_logical_seperation_value=True, feature_business_line_tpmb=True, feature_business_line_gtrm=False),
        "expected": ["IWPB", "FEMA", "TPMB"],
    },
    {
        "label": "REAL QUIRK 2: all 3 TPMB-tier roles present, but FEATURE_THIRD_PARTY_BOT off -- the shortcut doesn't fire, but TPMB still appears via the unconditional direct mapping (the flag only controls the early-return shortcut, not whether a held role maps at all)",
        "user_roles": ["stitt.ingester", "stitt.ingester.pilot", "stitt.ingester.tpmb"],
        "flags": dict(feature_logical_seperation_value=True, feature_business_line_tpmb=False, feature_business_line_gtrm=False),
        "expected": ["FEMA", "IWPB", "TPMB"],
    },
    {
        "label": "All 4 roles present, both TPMB and GTRM flags on -- TPMB shortcut fires FIRST (returns before GTRM check)",
        "user_roles": ["stitt.ingester", "stitt.ingester.pilot", "stitt.ingester.tpmb", "stitt.ingester.gtrm"],
        "flags": dict(feature_logical_seperation_value=True, feature_business_line_tpmb=True, feature_business_line_gtrm=True),
        "expected": ["IWPB", "FEMA", "TPMB"],
    },
    {
        "label": "REAL QUIRK: all 4 roles present, FEATURE_THIRD_PARTY_BOT OFF but FEATURE_GREEN_TIME ON -- GTRM shortcut still includes TPMB",
        "user_roles": ["stitt.ingester", "stitt.ingester.pilot", "stitt.ingester.tpmb", "stitt.ingester.gtrm"],
        "flags": dict(feature_logical_seperation_value=True, feature_business_line_tpmb=False, feature_business_line_gtrm=True),
        "expected": ["IWPB", "FEMA", "TPMB", "GTRM"],
    },
    {
        "label": "Only 3 of 4 roles present (gtrm role missing), FEATURE_GREEN_TIME on -- GTRM shortcut does NOT fire (subset check fails); TPMB still appears via direct mapping since the role is held",
        "user_roles": ["stitt.ingester", "stitt.ingester.pilot", "stitt.ingester.tpmb"],
        "flags": dict(feature_logical_seperation_value=True, feature_business_line_tpmb=False, feature_business_line_gtrm=True),
        "expected": ["FEMA", "IWPB", "TPMB"],
    },
    {
        "label": "No mapped roles at all -- empty list, not None (flag is on, just nothing maps)",
        "user_roles": ["some.other.unrelated.role"],
        "flags": dict(feature_logical_seperation_value=True, feature_business_line_tpmb=True, feature_business_line_gtrm=True),
        "expected": [],
    },
]

for i, scenario in enumerate(scenarios, start=1):
    result = role_check(scenario["user_roles"], **scenario["flags"])
    status = "OK" if result == scenario["expected"] else "MISMATCH"
    print(f"[{i}] {scenario['label']}")
    print(f"    roles={scenario['user_roles']}")
    print(f"    flags={scenario['flags']}")
    print(f"    -> {result}   (expected {scenario['expected']})   [{status}]")
    print()
    assert result == scenario["expected"], f"Scenario {i} failed: got {result}, expected {scenario['expected']}"

print("All scenarios matched the real /role-check algorithm's expected output.")


[1] Feature flag off entirely -- no business lines returned, regardless of roles
    roles=['stitt.ingester', 'stitt.ingester.pilot', 'stitt.ingester.tpmb', 'stitt.ingester.gtrm']
    flags={'feature_logical_seperation_value': False, 'feature_business_line_tpmb': True, 'feature_business_line_gtrm': True}
    -> None   (expected None)   [OK]

[2] IWPB-only role, flags on but TPMB/GTRM roles absent -- direct mapping only
    roles=['stitt.ingester']
    flags={'feature_logical_seperation_value': True, 'feature_business_line_tpmb': True, 'feature_business_line_gtrm': True}
    -> ['IWPB']   (expected ['IWPB'])   [OK]

[3] IWPB + FEMA roles, TPMB role missing -- direct mapping, alphabetically sorted
    roles=['stitt.ingester', 'stitt.ingester.pilot']
    flags={'feature_logical_seperation_value': True, 'feature_business_line_tpmb': True, 'feature_business_line_gtrm': True}
    -> ['FEMA', 'IWPB']   (expected ['FEMA', 'IWPB'])   [OK]

[4] All 3 TPMB-tier roles present, FEATURE_THIRD_PARTY_

## 3. Why scenarios 5, 7, and 8 are worth knowing cold in an interview

These three are genuine, source-confirmed quirks, not hypothetical edge cases:

- **Scenario 7**: because the real handler checks `FEATURE_THIRD_PARTY_BOT` and `FEATURE_GREEN_TIME` as
  two **independent** `if` blocks rather than an `if`/`elif` chain, a user holding all four
  `stitt.ingester*` roles sees `TPMB` in their dropdown the moment `FEATURE_GREEN_TIME` is turned on —
  even if `FEATURE_THIRD_PARTY_BOT` (the flag that's supposed to gate TPMB specifically) is still off.
- **Scenarios 5 and 8**: the unconditional `business_lines = [...]` direct mapping at the top of the
  function includes whatever the user's roles map to, with **no feature-flag check at all** — so a user
  individually holding `stitt.ingester.tpmb` sees `TPMB` in the fallback `sorted(business_lines)` result
  regardless of `FEATURE_THIRD_PARTY_BOT`. The flag's real, literal job is narrower than "gates TPMB" —
  it only gates the clean 3-role/4-role early-return shortcuts.

If asked "walk me through exactly how a user's roles become a list of visible business lines," being
able to name these precisely — not just "there's a mapping and some flags" — is a strong,
source-grounded signal that the answer comes from having actually read the code, not from a general
description of how RBAC + feature flags usually work.


## Summary

| What this notebook proved | How |
|---|---|
| `role_to_business_line` mapping matches `app.py` exactly | Direct port, same dict literal |
| `FEATURE_THIRD_PARTY_BOT`'s all-3-roles-required gate | Scenarios 4-5 |
| `FEATURE_GREEN_TIME`'s all-4-roles-required gate | Scenario 6 |
| TPMB-before-GTRM precedence (first matching `if` wins) | Scenario 6 |
| The real "GTRM flag can smuggle in TPMB" quirk | Scenario 7 |
| The direct mapping ignores feature flags entirely | Scenarios 5, 8 |
| `FEATURE_LOGICAL_SEPERATION` off returns `None`, not `[]` | Scenario 1 |
| No mapped roles returns `[]`, not `None` | Scenario 9 |

This is the exact algorithm chapter 03 and chapter 04 describe in prose — every branch in
`role_check()` above corresponds to a specific `if` block in the real `/role-check` handler in `app.py`.
